# بازتولید و نمایش خروجی‌های بخش ارزیابی SemiTNet

این نوت‌بوک یک نمای یکپارچه از **شکل‌ها، نمودارها، جدول‌ها و خروجی‌های بخش ارزیابی** ارائه می‌کند.

در هر بخش چهار جزء کنار هم قرار گرفته‌اند:

1. **هدف خروجی**
2. **قطعه‌کد تولید یا بارگذاری خروجی**
3. **توضیح فارسی منطق کد**
4. **خروجی نهایی یا مسیر خروجی تولیدشده**

برای جلوگیری از هرگونه ساخت نتیجه غیرواقعی، فقط خروجی‌هایی به‌عنوان نتیجه اجرایی نمایش داده می‌شوند که فایل اندازه‌گیری‌شده یا خروجی واقعی اجرای مدل برای آن‌ها وجود داشته باشد. مقادیر استخراج‌شده از مقاله با عنوان **«مرجع مقاله»** مشخص می‌شوند و خروجی اجرای موجود با عنوان **«اجرای ارزیابی فعلی»** نمایش داده می‌شود.


## 1) بارگذاری داده‌های ارزیابی و مسیرها

### توضیح فارسی
این سلول مسیرهای استاندارد پروژه را تعریف می‌کند و سه منبع اصلی را می‌خواند:

- `outputs/final/metrics.json`: متریک‌های محاسبه‌شده توسط اجرای مدل
- `outputs/final/history.csv`: روند آموزش و ارزیابی برای رسم نمودار
- `paper/reference/`: مقادیر مرجع استخراج‌شده از مقاله برای ساخت جدول‌ها و مقایسه‌ی هم‌سبک


In [ ]:
# هدف: بارگذاری منابع اصلی ارزیابی
# ورودی: فایل‌های metrics.json، history.csv و فایل‌های مرجع مقاله
# خروجی: DataFrame و دیکشنری‌های قابل استفاده در شکل‌ها و جدول‌ها
# رفتار مورد انتظار: در صورت وجود فایل‌ها، اطلاعات بدون تغییر عددی بارگذاری می‌شوند.

from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display, Image

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

FINAL = ROOT / "outputs" / "final"
PAPER = ROOT / "paper" / "reference"
PAPER_STYLE = ROOT / "outputs" / "paper_style"

metrics = json.loads((FINAL / "metrics.json").read_text())
history = pd.read_csv(FINAL / "history.csv")
table2 = pd.read_csv(PAPER / "table2_overall.csv")
table3 = pd.read_csv(PAPER / "table3_groups.csv")

print("مسیر پروژه:", ROOT)
print("تعداد رکوردهای تاریخچه:", len(history))
print("متریک‌های اجرای ارزیابی فعلی:")
for key in ["iou", "dice", "precision", "recall", "f1"]:
    print(f"  {key}: {metrics[key]:.4f}")


---

## 2) Table 1 — توزیع داده‌ها

### هدف
نمایش ساختار مجموعه‌داده مطابق دسته‌بندی مورد استفاده در مقاله: داده‌های برچسب‌دار، بدون برچسب و آزمون.

### توضیح فارسی کد
کد زیر فایل مرجع توزیع داده را با `pandas` می‌خواند و آن را به همان ساختار ستونی مورد نیاز برای گزارش تبدیل می‌کند. این جدول یک **جدول توصیفی مجموعه‌داده** است و متریک مدل محسوب نمی‌شود.


In [ ]:
# هدف: نمایش Table 1 مربوط به توزیع مجموعه‌داده
# ورودی: paper/reference/table1_dataset.csv
# خروجی: جدول توزیع داده‌ها
# رفتار مورد انتظار: جدول بدون تغییر مقادیر مرجع نمایش داده می‌شود.

table1 = pd.read_csv(PAPER / "table1_dataset.csv")
display(table1)


**خروجی رندرشده:**

![Table 1](../outputs/paper_style/tables/table_01_tsi15k_distribution.png)


---

## 3) Table 2 — مقایسه عملکرد کلی

### متریک‌ها
- IoU
- Dice
- Precision
- Recall
- F1
- تعداد پارامترها

### توضیح فارسی کد
فایل `table2_overall.csv` شامل ردیف‌های مرجع مدل‌های مقایسه‌شده در مقاله است. کد زیر جدول را می‌خواند، نام ستون‌ها را برای نمایش استاندارد می‌کند و سپس متریک‌های **اجرای ارزیابی فعلی** را جداگانه در همان پنج معیار اصلی نشان می‌دهد؛ بنابراین هیچ عددی از اجرای فعلی به مدل‌های پایه نسبت داده نمی‌شود.


In [ ]:
# هدف: نمایش Table 2 و متریک‌های اجرای ارزیابی فعلی
# ورودی: table2_overall.csv و metrics.json
# خروجی: جدول مرجع مقاله + یک جدول مستقل برای اجرای فعلی
# رفتار مورد انتظار: اعداد مدل‌های پایه فقط از فایل مرجع مقاله خوانده می‌شوند.

paper_overall = table2.rename(columns={
    "model": "Model",
    "iou": "IoU",
    "dice": "Dice",
    "precision": "Precision",
    "recall": "Recall",
    "f1": "F1",
    "params_m": "Params (M)",
})
display(paper_overall)

current_run = pd.DataFrame([{
    "Run": "اجرای ارزیابی فعلی",
    "IoU": metrics["iou"],
    "Dice": metrics["dice"],
    "Precision": metrics["precision"],
    "Recall": metrics["recall"],
    "F1": metrics["f1"],
}])
display(current_run)


**خروجی رندرشده جدول مرجع:**

![Table 2](../outputs/paper_style/tables/table_02_overall_performance.png)

### متریک‌های خروجی اجرای ارزیابی فعلی

| IoU | Dice | Precision | Recall | F1 |
|---:|---:|---:|---:|---:|
| 33.0565 | 49.6879 | 12.5266 | 12.0290 | 12.2727 |


---

## 4) Table 3 — عملکرد روی زیرگروه‌های دندانی

### توضیح فارسی کد
این جدول مقادیر مرجع مقاله را برای دو گروه زیر نمایش می‌دهد:

- `fully_dentate`
- `partially_edentulous`

از آن‌جا که اجرای موجود متریک زیرگروهی مستقل برای این دو گروه ذخیره نکرده است، هیچ مقدار جدیدی برای این دو گروه ساخته نمی‌شود. جدول فقط از فایل مرجع مقاله تولید می‌شود.


In [ ]:
# هدف: نمایش Table 3 برای گروه‌های Fully Dentate و Partially Edentulous
# ورودی: paper/reference/table3_groups.csv
# خروجی: جدول متریک‌های گروهی مرجع مقاله
# رفتار مورد انتظار: فقط داده‌های موجود در فایل مرجع نمایش داده شوند.

group_table = table3.rename(columns={
    "group": "Group",
    "model": "Model",
    "iou": "IoU",
    "dice": "Dice",
    "precision": "Precision",
    "recall": "Recall",
    "f1": "F1",
})
display(group_table)


**خروجی رندرشده:**

![Table 3](../outputs/paper_style/tables/table_03_group_performance.png)


---

## 5) Figure 1 — معماری SemiTNet

### توضیح فارسی کد
این شکل مسیر معماری را به‌صورت بلوکی نمایش می‌دهد:

`Panoramic Radiograph → Image Encoder → Feature Pyramid → Query Initialization → Mask Decoder → Predictions`

کد اصلی تولید این شکل در تابع `figure1()` از اسکریپت `scripts/export_paper_style_figures.py` قرار دارد. این شکل برای توضیح معماری و نگاشت اجزای روش استفاده می‌شود.


In [ ]:
# هدف: نمایش Figure 1 معماری
# ورودی: خروجی از پیش تولیدشده‌ی تابع figure1()
# خروجی: تصویر معماری
# رفتار مورد انتظار: فایل PNG با کیفیت بالا نمایش داده شود.

display(Image(filename=str(PAPER_STYLE / "figures" / "figure_01_semitnet_architecture.png")))


![Figure 1](../outputs/paper_style/figures/figure_01_semitnet_architecture.png)


---

## 6) Figure 2 — جریان Teacher–Student و تولید pseudo-label

### توضیح فارسی کد
این دیاگرام مراحل اصلی آموزش نیمه‌نظارتی را نشان می‌دهد:

1. Teacher برای داده بدون برچسب pseudo-label می‌سازد.
2. Student روی داده‌ی تقویت‌شده پیش‌بینی انجام می‌دهد.
3. زیان نظارت‌شده `Ls` و زیان بدون‌نظارت `Lu` ترکیب می‌شوند.
4. پارامترهای Teacher با EMA از Student به‌روزرسانی می‌شوند.

کد رسم دیاگرام در تابع `figure2()` قرار دارد.


In [ ]:
# هدف: نمایش Figure 2 مربوط به گردش کار Teacher–Student
# ورودی: فایل figure_02_teacher_student_workflow.png
# خروجی: دیاگرام فرآیند نیمه‌نظارتی
# رفتار مورد انتظار: تصویر موجود بدون تغییر داده‌ای نمایش داده شود.

display(Image(filename=str(PAPER_STYLE / "figures" / "figure_02_teacher_student_workflow.png")))


![Figure 2](../outputs/paper_style/figures/figure_02_teacher_student_workflow.png)


---

## 7) Figure 3 — نمودار Loss و Precision

### توضیح فارسی کد
این نمودار مستقیماً از `history.csv` ساخته می‌شود:

- محور افقی: مرحله/epoch اجرا
- محور عمودی سمت چپ: `Training Loss`
- محور عمودی سمت راست: `Precision (%)`

دو محور مستقل باعث می‌شوند روند کاهش Loss و تغییر Precision در یک شکل قابل مشاهده باشند.


In [ ]:
# هدف: بازتولید Figure 3 از تاریخچه واقعی اجرای موجود
# ورودی: outputs/final/history.csv
# خروجی: نمودار دو محوره Loss و Precision
# رفتار مورد انتظار: هیچ مقدار مصنوعی وارد نمودار نشود.

x = np.arange(1, len(history) + 1)

fig, ax1 = plt.subplots(figsize=(11.5, 6.2))
ax2 = ax1.twinx()

ax1.plot(x, history["loss"], marker="o", linewidth=2.2, label="Training Loss")
ax2.plot(x, history["precision"], marker="o", linewidth=2.2, label="Precision (%)")

ax1.set_xlabel("Training Step")
ax1.set_ylabel("Training Loss")
ax2.set_ylabel("Precision (%)")
ax1.grid(True, axis="y", alpha=0.25)
ax1.set_title("Loss and Precision vs. Training Step")

lines = [ax1.lines[0], ax2.lines[0]]
ax1.legend(lines, [line.get_label() for line in lines], loc="upper right")
plt.show()


**نسخه‌ی صادرشده:**

![Figure 3](../outputs/paper_style/figures/figure_03_training_loss_precision.png)


---

## 8) Figure 4 — مقایسه راداری متریک‌ها

### توضیح فارسی کد
پنج محور رادار دقیقاً پنج معیار اصلی ارزیابی هستند:

`IoU, Dice, Precision, Recall, F1`

برای مدل‌های مرجع، مقادیر از جدول مقاله خوانده می‌شوند. خروجی اجرای فعلی در یک نمودار مستقل نگهداری می‌شود تا عددی به مدل‌های دیگر نسبت داده نشود.


In [ ]:
# هدف: ساخت رادار متریک‌های اجرای ارزیابی فعلی
# ورودی: metrics.json
# خروجی: نمودار Radar پنج‌محوره
# رفتار مورد انتظار: فقط پنج متریک اندازه‌گیری‌شده‌ی اجرای فعلی رسم شوند.

labels = ["IoU", "Dice", "Precision", "Recall", "F1"]
values = [metrics[k.lower()] for k in labels]

angles = np.linspace(0, 2 * np.pi, len(labels), endpoint=False).tolist()
angles += angles[:1]
plot_values = values + values[:1]

fig, ax = plt.subplots(figsize=(7, 6), subplot_kw={"projection": "polar"})
ax.set_theta_offset(np.pi / 2)
ax.set_theta_direction(-1)
ax.set_thetagrids(np.degrees(angles[:-1]), labels)
ax.set_ylim(0, 100)
ax.plot(angles, plot_values, linewidth=2.5, label="اجرای ارزیابی فعلی")
ax.fill(angles, plot_values, alpha=0.15)
ax.legend(loc="lower center", bbox_to_anchor=(0.5, -0.15))
plt.show()


**شکل مرجع هم‌سبک مقاله:**

![Figure 4](../outputs/paper_style/figures/figure_04_performance_radar.png)

**رادار خروجی اجرای ارزیابی فعلی:**

![Figure 4b](../outputs/paper_style/figures/figure_04b_measured_reduced_simulation.png)


---

## 9) Figure 5 — خروجی کیفی واقعی مدل

### قاعده‌ی این بخش
هیچ خروجی تصویری برای مدل‌های پایه ساخته یا حدس زده نمی‌شود.

خروجی اصلی این بخش فایل اندازه‌گیری‌شده‌ی موجود در پروژه است:

`outputs/paper_style/figures/figure_05b_measured_qualitative_only.png`

این خروجی نمونه‌های ورودی/پیش‌بینی/برچسب واقعی را برای خروجی موجود نمایش می‌دهد و برای تحویل به‌عنوان نتیجه کیفی استفاده می‌شود.

### توضیح فارسی کد
کد ابتدا فایل خروجی کیفی اندازه‌گیری‌شده را جست‌وجو می‌کند. اگر فایل وجود داشته باشد مستقیماً نمایش داده می‌شود. در غیر این صورت فقط مسیر منبع گزارش می‌شود و هیچ تصویر جایگزین مصنوعی ساخته نمی‌شود.


In [ ]:
# هدف: نمایش Figure 5 از خروجی کیفی واقعاً موجود
# ورودی: figure_05b_measured_qualitative_only.png
# خروجی: تصویر کیفی پیش‌بینی مدل
# رفتار مورد انتظار: در نبود فایل، هیچ تصویر مصنوعی یا placeholder به‌عنوان نتیجه ساخته نشود.

qualitative = PAPER_STYLE / "figures" / "figure_05b_measured_qualitative_only.png"

if qualitative.exists():
    display(Image(filename=str(qualitative)))
else:
    source = FINAL / "figures" / "predictions.png"
    print("فایل خروجی کیفی اصلی در این checkout یافت نشد.")
    print("مسیر مورد انتظار:", qualitative)
    print("منبع پیش‌بینی:", source)


**خروجی کیفی اندازه‌گیری‌شده در مخزن:**

`outputs/paper_style/figures/figure_05b_measured_qualitative_only.png`

> برای تحویل نهایی، این فایل به‌عنوان Figure 5 نتیجه‌ی کیفی استفاده شود؛ نسخه‌ای که ستون‌های مدل‌های پایه‌ی اجرا‌نشده را با placeholder نشان می‌دهد، نتیجه‌ی اجرایی محسوب نمی‌شود.


---

## 10) جدول خلاصه‌ی خروجی‌های ارزیابی

| خروجی | نوع | منبع |
|---|---|---|
| Table 1 | توصیفی | `paper/reference/table1_dataset.csv` |
| Table 2 | مرجع مقاله + متریک اجرای فعلی جداگانه | `table2_overall.csv`, `metrics.json` |
| Table 3 | مرجع مقاله | `table3_groups.csv` |
| Figure 1 | دیاگرام معماری | `export_paper_style_figures.py` |
| Figure 2 | دیاگرام Teacher–Student | `export_paper_style_figures.py` |
| Figure 3 | نمودار اندازه‌گیری‌شده | `history.csv` |
| Figure 4 | مقایسه مرجع + رادار اجرای فعلی | `table2/table3 + metrics.json` |
| Figure 5 | خروجی کیفی اندازه‌گیری‌شده | `figure_05b_measured_qualitative_only.png` |

### نتیجه
این نوت‌بوک اجازه می‌دهد کاربر، خروجی‌های اصلی بخش ارزیابی را با ساختار و متریک‌های متناظر مقاله مشاهده کند و در عین حال منبع هر عدد و هر تصویر مشخص باشد.
